# Weighted Least Squares: the idea in plain language

Weighted Least Squares (WLS) fits a model to measurements while taking their reliability into account. Ordinary least squares treats every measurement equally. WLS gives **more influence to accurate measurements** and **less influence to noisy measurements**.

## The measurement model

We assume that each measurement can be described by

$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{v}
$$

- \(\mathbf{y}\): the measurements we observed.
- \(\mathbf{H}\): the model matrix, which describes how the unknowns affect each measurement.
- \(\mathbf{x}\): the unknown parameters we want to estimate.
- \(\mathbf{v}\): measurement error or noise.

The covariance matrix \(\mathbf{R}\) describes the uncertainty of the measurements. For independent measurements, it is diagonal:

$$
\mathbf{R} = \operatorname{diag}(\sigma_1^2, \sigma_2^2, \ldots, \sigma_n^2)
$$

Here \(\sigma_i\) is the standard deviation of measurement \(i\). A large \(\sigma_i\) means that measurement is less certain. WLS uses \(\mathbf{R}^{-1}\) as the weighting matrix, so the weight of an independent measurement is

$$
w_i = \frac{1}{\sigma_i^2}.
$$

For example, a measurement with \(\sigma=1\) has weight \(1\), while one with \(\sigma=2\) has weight \(1/4\). The second measurement is therefore treated as four times less reliable.

## The WLS estimate

WLS chooses the parameter vector that best fits the measurements after accounting for their uncertainties:

$$
\hat{\mathbf{x}} = (\mathbf{H}^T\mathbf{R}^{-1}\mathbf{H})^{-1}\mathbf{H}^T\mathbf{R}^{-1}\mathbf{y}.
$$

The covariance of this estimate is

$$
\mathbf{P} = (\mathbf{H}^T\mathbf{R}^{-1}\mathbf{H})^{-1}.
$$

In this notebook, the model is a straight line relating temperature \(y\) to RPM \(r\):

$$\hat{y} = \hat{x}_1 r + \hat{x}_2,$$

where \(\hat{x}_1\) is the estimated slope and \(\hat{x}_2\) is the estimated intercept. Each dataset row is stored as \([y_i, r_i, \sigma_i]\). The next section derives the corresponding \(\mathbf{y}\), \(\mathbf{H}\), and \(\mathbf{R}\) matrices.

## Mathematical Explanation

In linear least squares estimation, we want to find a parameter vector $x$ that best fits a linear model of the form:

$$y = Hx + v$$

Where:

- $y$ is the measurement vector (dependent variable, e.g., temperature).
- $H$ is the model/design matrix (independent variables/basis functions, e.g., RPM values and a column of ones for the intercept).
- $x$ is the constant vector of parameters we want to estimate (e.g., slope $x_1$ and intercept $x_2$).
- $v$ is the measurement noise.

The analytical solution that minimizes the sum of the squared residuals is given by the normal equations:

$$x = (H^T H)^{-1} H^T y$$

Using the hints provided, this can be implemented in NumPy using np.matmul(), np.transpose(), and np.linalg.inv().

For the line of best fit ($y_i = x_1 \cdot r_i + x_2$), each data point provides a row in our system:

- The model matrix $H$ has two columns: the first column contains the RPM values ($r_i$), and the second column contains ones (for the intercept $x_2$).
- The measurement vector $Y$ contains the temperature values ($y_i$).
---

In [14]:
import numpy as np

Dataset = [
    [62, 1, 1],
    [68, 2, 2],
    [74, 3, 1.5],
    [82, 4, 1],
    [89, 5, 2],
    [95, 6, 1],
    [103, 7, 1.5],
    [109, 8, 2]
]

In [15]:
def CalculateLeastSquaresSolution(HMatrix, YMatrix):
    """
    Calculates the general LSE solution x given H and Y
    Formula: x = (H^T * H)^-1 * H^T * Y
    """
    # Convert inputs to numpy arrays to ensure matrix operations
    H = np.array(HMatrix)
    Y = np.array(YMatrix)

    # Transpose of H Matrix 
    H_T = np.transpose(H)

    # Calculate (H^T, H)
    HtH = np.malmut(H_T,H)

    # Calculate the inverse of (H^T * H)^-1
    HtH_inverse = np.linalg.inv(HtH)

    # Calculate (H^T * Y)
    HtY = np.malmut(H_T,Y)

    # Calculate the final parameter vector X = (H^T * H)^-1*H^T*Y

    Xmatrix = np.malmut(HtH_inverse,HtY)

    return Xmatrix

def CalculateLineOfBestFitSolution(Dataset):
    """
    Generates the H and Y matrices from the Dataset {[y_1, r_1], [y_2, r_2], ..., [y_n, r_n]}
    and calculates the line parameters [x_1 (slope), x_2 (intercept)].
    Model: y_i = x_1 * r_i + x_2
    """

    Hmatrix= []
    Ymatrix = []

    for row in Dataset: 
        y_i = row[0]
        r_i = row[1]

        # Y vector contains temperature measurements (y_i)
        Ymatrix.append([y_i])

        # H matrix row contains [r_i, 1] to solve for x_1 (slope)
        Hmatrix.append([r_i, 1.0])

    LineParam = CalculateLeastSquaresSolution(Hmatrix, Ymatrix)

    # Flatten or return the params
    return LineParam


# Weighted Least Squares Estimation

## 1. Introduction

Weighted Least Squares (WLS) is an extension of Ordinary Least Squares (OLS). It is useful when measurements do not all have the same accuracy.

Ordinary Least Squares treats every measurement equally. Weighted Least Squares gives more influence to reliable measurements and less influence to uncertain measurements.

The measurement model is

$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{v}
$$

where:

- $\mathbf{y}$ is the vector of measured values.
- $\mathbf{H}$ is the model or design matrix.
- $\mathbf{x}$ is the unknown parameter vector.
- $\mathbf{v}$ represents measurement errors or noise.

The measurement uncertainty is described by the covariance matrix

$$
\mathbf{R} = E[\mathbf{v}\mathbf{v}^T].
$$

A larger value in $\mathbf{R}$ means that the corresponding measurement is less reliable.

## 2. Why Use Weighted Least Squares?

Suppose two measurements have standard deviations

$$
\sigma_1 = 1,
\qquad
\sigma_2 = 3.
$$

The first measurement is more accurate because it has a smaller standard deviation.

For independent measurements, the weight of measurement $i$ is

$$
w_i = \frac{1}{\sigma_i^2}.
$$

Therefore,

$$
w_1 = \frac{1}{1^2} = 1,
\qquad
w_2 = \frac{1}{3^2} = \frac{1}{9}.
$$

The second measurement receives a much smaller weight because it is more uncertain.

## 3. Covariance and Weighting Matrices

If measurement errors are independent, the covariance matrix is diagonal:

$$
\mathbf{R} =
\begin{bmatrix}
\sigma_1^2 & 0 & \cdots & 0 \\
0 & \sigma_2^2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \sigma_n^2
\end{bmatrix}.
$$

The inverse covariance matrix is

$$
\mathbf{R}^{-1} =
\begin{bmatrix}
\dfrac{1}{\sigma_1^2} & 0 & \cdots & 0 \\
0 & \dfrac{1}{\sigma_2^2} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \dfrac{1}{\sigma_n^2}
\end{bmatrix}.
$$

The matrix $\mathbf{R}^{-1}$ is used as the weighting matrix. Measurements with smaller uncertainty receive larger weights.

## 4. Weighted Least Squares Solution

The WLS estimate is

$$
\hat{\mathbf{x}}
=
\left(
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{H}
\right)^{-1}
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{y}.
$$

Here:

- $\mathbf{H}^T$ is the transpose of $\mathbf{H}$.
- $\mathbf{R}^{-1}$ contains the measurement weights.
- $\hat{\mathbf{x}}$ is the estimated parameter vector.

The covariance of the estimated parameters is

$$
\mathbf{P}
=
\left(
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{H}
\right)^{-1}.
$$

This matrix describes the uncertainty of the estimated parameters.

## 5. Straight-Line Model

In this example, we estimate the relationship between temperature and RPM using a straight line:

$$
y_i = x_1 r_i + x_2
$$

where:

- $y_i$ is the measured temperature.
- $r_i$ is the RPM value.
- $x_1$ is the slope.
- $x_2$ is the intercept.

The unknown parameter vector is

$$
\mathbf{x}
=
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}.
$$

For one measurement,

$$
y_i =
\begin{bmatrix}
r_i & 1
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
+ v_i.
$$

For multiple measurements,

$$
\mathbf{y}
=
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_n
\end{bmatrix},
\qquad
\mathbf{H}
=
\begin{bmatrix}
r_1 & 1 \\
r_2 & 1 \\
\vdots & \vdots \\
r_n & 1
\end{bmatrix}.
$$

The complete model is

$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{v}.
$$

## 6. Dataset Structure

Each row of the dataset has the form

```python
[y_i, r_i, sigma_i]

In [16]:
import numpy as np

def CalculateWeightedLeastSquaresSolution(Hmatrix, Rmatrix, Ymatrix):

    Hmatrix = np.array(Hmatrix)
    Rmatrix = np.array(Rmatrix)
    Ymatrix = np.array(Ymatrix)

    # Transpose of H 
    Htranspose = np.transpose(Hmatrix)

    # Inverse of R 
    Rinverse = np.linalg.inv(Rmatrix)

    # Parameter covariance matrix: 
    # P = (H^T R^-1 H)^-1 
    Pmatrix = np.linalg.inv(np.matmul(np.matmul(Htranspose, Rinverse), Hmatrix))

    # Weighted least squares solution: 
    # X = P H^T R^-1 Y 
    Xmatrix = np.matmul(np.matmul(np.matmul(Pmatrix, Htranspose), Rinverse), Ymatrix)

    return Xmatrix, Pmatrix

def CalculateLineOfBestFitSolution(Dataset):

    #Generate H, R and Y matrices 
    Hmatrix = []
    Rmatrix = []
    Ymatrix = []

    n = len(Dataset)
    print(n)

    for measurement in Dataset: 

        y = measurement[0]
        print(y)
        r = measurement[1]
        print(r)

        # Model: y_i = x1*r_i + x2
        Hmatrix.append(y)

        #IMPORTANT: append y, not [y]
        Ymatrix.append(y)

    # Create covariance matrix 
    Rmatrix = np.zeros((n,n))

    for i in range(n): 

        sigma = Dataset[i][2]

        # Variance = sigma^2
        Rmatrix[i][i] = sigma ** 2

    Lineparam, LineParamCov = CalculateWeightedLeastSquaresSolution(Hmatrix, Rmatrix, Ymatrix)
    return Lineparam



# Recursive Least Squares Estimation

## 1. Introduction

Recursive Least Squares (RLS) estimates unknown model parameters one measurement at a time.

Unlike ordinary or weighted least squares, RLS does not need to store all previous measurements or recompute the complete solution. Instead, it updates the previous estimate whenever a new measurement becomes available.

The measurement model at iteration $k$ is

$$
\mathbf{y}_k = \mathbf{H}_k\mathbf{x}_{k-1} + \mathbf{v}_k
$$

where:

- $\mathbf{y}_k$ is the new measurement.
- $\mathbf{H}_k$ is the model matrix for that measurement.
- $\mathbf{x}_{k-1}$ is the parameter estimate from the previous iteration.
- $\mathbf{v}_k$ is the measurement noise.
- $\mathbf{R}_k$ is the covariance or variance of the new measurement.

The previous estimate has covariance matrix $\mathbf{P}_{k-1}$, which represents the uncertainty in the estimated parameters.

## 2. Predicting the Measurement

Before using the new measurement, RLS predicts its value using the previous parameter estimate:

$$
\hat{\mathbf{y}}_k
=
\mathbf{H}_k\mathbf{x}_{k-1}.
$$

The difference between the actual and predicted measurements is called the innovation or residual:

$$
\mathbf{e}_k
=
\mathbf{y}_k - \hat{\mathbf{y}}_k
=
\mathbf{y}_k - \mathbf{H}_k\mathbf{x}_{k-1}.
$$

A small residual means that the new measurement agrees with the previous estimate. A large residual indicates that the estimate may need a larger correction.

## 3. Innovation Covariance

The uncertainty of the prediction is described by the innovation covariance:

$$
\mathbf{S}_k
=
\mathbf{H}_k\mathbf{P}_{k-1}\mathbf{H}_k^T
+
\mathbf{R}_k.
$$

The first term represents uncertainty in the previous parameter estimate. The second term represents uncertainty in the new measurement.

## 4. Recursive Gain

The recursive gain determines how strongly the new measurement affects the estimate:

$$
\mathbf{K}_k
=
\mathbf{P}_{k-1}\mathbf{H}_k^T\mathbf{S}_k^{-1}.
$$

A measurement with low uncertainty produces a larger correction. A measurement with high uncertainty produces a smaller correction.

## 5. Updating the Parameter Estimate

The parameter estimate is updated using the residual:

$$
\mathbf{x}_k
=
\mathbf{x}_{k-1}
+
\mathbf{K}_k\mathbf{e}_k.
$$

Substituting the residual gives

$$
\mathbf{x}_k
=
\mathbf{x}_{k-1}
+
\mathbf{K}_k
\left(
\mathbf{y}_k
-
\mathbf{H}_k\mathbf{x}_{k-1}
\right).
$$

In words:

> New estimate = previous estimate + correction from the new measurement.

## 6. Updating the Parameter Covariance

After processing the measurement, the parameter covariance is updated using

$$
\mathbf{P}_k
=
\left(
\mathbf{I}
-
\mathbf{K}_k\mathbf{H}_k
\right)
\mathbf{P}_{k-1}.
$$

The covariance generally decreases as more informative measurements are processed. This means that the uncertainty of the parameter estimate becomes smaller.

## 7. Straight-Line Model

In this example, the relationship between temperature and RPM is modeled as

$$
y_k = x_1 r_k + x_2 + v_k
$$

where:

- $y_k$ is the measured temperature.
- $r_k$ is the RPM value.
- $x_1$ is the slope.
- $x_2$ is the intercept.
- $v_k$ is measurement noise.

The parameter vector is

$$
\mathbf{x}_k
=
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}.
$$

For one measurement, the model matrix is

$$
\mathbf{H}_k
=
\begin{bmatrix}
r_k & 1
\end{bmatrix}.
$$

Therefore,

$$
\mathbf{y}_k
=
\begin{bmatrix}
y_k
\end{bmatrix},
\qquad
\mathbf{H}_k
=
\begin{bmatrix}
r_k & 1
\end{bmatrix}.
$$

The matrix $\mathbf{H}_k$ has shape $1 \times 2$, while $\mathbf{x}_k$ has shape $2 \times 1$.

## 8. Dataset Structure

Each new measurement is stored as

```python
[y_i, r_i, variance_i]

In [25]:
import numpy as np 

def CalculateRecursiveLeastSquaresSolution(Xmatrix, Pmatrix, Hmatrix, Rmatrix, Ymatrix):

    # Convert all inputs to Numpy arrays 
    Xmatrix = np.array(Xmatrix)
    Pmatrix = np.array(Pmatrix)
    Hmatrix = np.array(Hmatrix)
    Rmatrix = np.array(Rmatrix)
    Ymatrix = np.array(Ymatrix)

    # Transpose of H 
    Htranspose = np.transpose(Hmatrix)

    # Calculate the innovation covariance: 
    # S = H P H^T + R 
    Smatrix = (np.matmul(np.matmul(Hmatrix, Pmatrix),Htranspose) + Rmatrix)

    # Calculate the recursive least squares gain: 
    # K = P H^T S^-1

    Kmatrix = np.matmul(np.matmul(Pmatrix, Htranspose),np.linalg.inv(Smatrix))

    # Calculate the measurement residual: 
    # residual = Y - H X
    Residual = Ymatrix - np.matmul(Hmatrix, Xmatrix)

    # Update the parameter estimate: 
    # X_k = X_(k-1) + K residual 
    Xmatrix = Xmatrix + np.matmul(Kmatrix, Residual)

    # Identity matrix 
    Imatrix = np.identity(Pmatrix.shape[0])

    # Update covariance_
    # P_k = (I - K H) P_(k-1)
    Pmatrix = np.matmul(Imatrix - np.matmul(Kmatrix, Hmatrix), Pmatrix)

    return Xmatrix, Pmatrix

def CalculateLineOfBestFitSolution(LineParam, LineParamCov, Dataset): 

    # Dataset = [temperature, RPM, variance]
    y = Dataset[0]
    r = Dataset[1]
    sigma_squared = Dataset[2]

    # Model: 
    # y_i = x1*r_i + x2
    #
    # H must be a row vector
    Hmatrix = np.array([[r,1]])

    # R is the measurement variance 
    # The dataset already contains sigma^2
    # therefore DO NO square it again 
    Rmatrix = np.array([[sigma_squared]])

    # Y is represented as a 1x1 array 
    Ymatrix = np.array([[y]])

    # Perform one recursive least squares update
    LineParam, LineParamCov = CalculateRecursiveLeastSquaresSolution(LineParam, LineParamCov, Hmatrix, Rmatrix, Ymatrix)

    return LineParam, LineParamCov
